In [120]:
import torch
from torch_geometric.datasets import Planetoid

# Load the CORA dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')


In [121]:
data = dataset[0]
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [122]:
print(f'Dataset: {dataset}')
print(f"Nodes (papers):        {data.num_nodes}")
print(f"Edges (citations):     {data.num_edges}")
print(f'Number of Classes (topics): {dataset.num_classes}')
print(f"Node feature dim:      {dataset.num_node_features}  (bag-of-words vocab)")
print(f"Training nodes:        {int(data.train_mask.sum())}  "
      f"({100*int(data.train_mask.sum())/data.num_nodes:.1f}% of all nodes labeled)")
print(f"Validation nodes:      {int(data.val_mask.sum())}")
print(f"Test nodes:            {int(data.test_mask.sum())}")


Dataset: Cora()
Nodes (papers):        2708
Edges (citations):     10556
Number of Classes (topics): 7
Node feature dim:      1433  (bag-of-words vocab)
Training nodes:        140  (5.2% of all nodes labeled)
Validation nodes:      500
Test nodes:            1000


In [124]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Layer 1: message pass + aggregate + update, then non-linearity
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Layer 2: project down to class logits
        x = self.conv2(x, edge_index)
        return x


In [125]:
def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, mask):
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    correct = (pred[mask] == data.y[mask]).sum()
    return int(correct) / int(mask.sum())

In [126]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

In [140]:
class TraditionalNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.layer1 = torch.nn.Linear(in_channels, hidden_channels)
        self.layer2 = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x = data.x
        x = self.layer1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.layer2(x)
        return x

# Training GCN with 16 channels

In [146]:
epochs = 200
model = GCN(
    in_channels=dataset.num_node_features,
    hidden_channels=16, 
    out_channels=dataset.num_classes
    ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print("=== Training ===")
for epoch in range(epochs):
    loss = train(model, optimizer)
    val_acc = evaluate(model, data.val_mask)
    print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")


test_acc = evaluate(model, data.test_mask)
print(f"=== Final Test Accuracy: {test_acc:.4f} ===")

=== Training ===
Epoch   0 | Loss 1.9411 | Val Acc 0.4620
Epoch   1 | Loss 1.8442 | Val Acc 0.5680
Epoch   2 | Loss 1.7180 | Val Acc 0.6100
Epoch   3 | Loss 1.5889 | Val Acc 0.6160
Epoch   4 | Loss 1.4437 | Val Acc 0.6200
Epoch   5 | Loss 1.2991 | Val Acc 0.6560
Epoch   6 | Loss 1.1542 | Val Acc 0.6960
Epoch   7 | Loss 1.0834 | Val Acc 0.7220
Epoch   8 | Loss 0.9387 | Val Acc 0.7440
Epoch   9 | Loss 0.8438 | Val Acc 0.7500
Epoch  10 | Loss 0.7382 | Val Acc 0.7560
Epoch  11 | Loss 0.7162 | Val Acc 0.7620
Epoch  12 | Loss 0.6037 | Val Acc 0.7660
Epoch  13 | Loss 0.5318 | Val Acc 0.7660
Epoch  14 | Loss 0.4649 | Val Acc 0.7680
Epoch  15 | Loss 0.4310 | Val Acc 0.7680
Epoch  16 | Loss 0.3828 | Val Acc 0.7680
Epoch  17 | Loss 0.3410 | Val Acc 0.7660
Epoch  18 | Loss 0.2888 | Val Acc 0.7720
Epoch  19 | Loss 0.2821 | Val Acc 0.7700
Epoch  20 | Loss 0.2391 | Val Acc 0.7660
Epoch  21 | Loss 0.2372 | Val Acc 0.7700
Epoch  22 | Loss 0.2212 | Val Acc 0.7720
Epoch  23 | Loss 0.1793 | Val Acc 0.7700

# Training Linear NN with 16 channels

In [144]:
epochs = 200
model = TraditionalNN(
    in_channels=dataset.num_node_features,
    hidden_channels=16, 
    out_channels=dataset.num_classes
    ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print("=== Training Traditional NN ===")
for epoch in range(epochs):
    loss = train(model, optimizer)
    val_acc = evaluate(model, data.val_mask)
    print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")


test_acc = evaluate(model, data.test_mask)
print(f"=== Final Test Accuracy: {test_acc:.4f} ===")

=== Training Traditional NN ===
Epoch   0 | Loss 1.9541 | Val Acc 0.1320
Epoch   1 | Loss 1.9195 | Val Acc 0.1700
Epoch   2 | Loss 1.8659 | Val Acc 0.2420
Epoch   3 | Loss 1.7995 | Val Acc 0.2800
Epoch   4 | Loss 1.6700 | Val Acc 0.2920
Epoch   5 | Loss 1.5705 | Val Acc 0.2940
Epoch   6 | Loss 1.4667 | Val Acc 0.2940
Epoch   7 | Loss 1.3724 | Val Acc 0.3120
Epoch   8 | Loss 1.2436 | Val Acc 0.3300
Epoch   9 | Loss 1.1973 | Val Acc 0.3500
Epoch  10 | Loss 1.0914 | Val Acc 0.3580
Epoch  11 | Loss 0.9572 | Val Acc 0.3840
Epoch  12 | Loss 0.9041 | Val Acc 0.4020
Epoch  13 | Loss 0.7714 | Val Acc 0.4180
Epoch  14 | Loss 0.7953 | Val Acc 0.4340
Epoch  15 | Loss 0.7359 | Val Acc 0.4580
Epoch  16 | Loss 0.6649 | Val Acc 0.4680
Epoch  17 | Loss 0.5808 | Val Acc 0.4800
Epoch  18 | Loss 0.5543 | Val Acc 0.4820
Epoch  19 | Loss 0.5173 | Val Acc 0.4940
Epoch  20 | Loss 0.5268 | Val Acc 0.5020
Epoch  21 | Loss 0.4610 | Val Acc 0.5080
Epoch  22 | Loss 0.4643 | Val Acc 0.5040
Epoch  23 | Loss 0.4204 |

# Comparing results with different channel configs

In [135]:
epochs = 200
result = []
hidden_channels = [8, 16, 32, 64, 128]
print("=== Training ===")
for hidden in hidden_channels:
    model = GCN(
        in_channels=dataset.num_node_features,
        hidden_channels=hidden, 
        out_channels=dataset.num_classes
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for epoch in range(epochs):
        loss = train(model, optimizer)
        if (epoch+1) % 100 == 0:
                val_acc = evaluate(model, data.val_mask)
                print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")    
    test_acc = evaluate(model, data.test_mask)
    print(f"=== Channel={hidden}, Final Test Accuracy: {test_acc:.4f} ===")
    output = {'hidden_channels': hidden, 'validation_accuracy':val_acc, 'test_accuracy': test_acc}
    result.append(output)


=== Training ===
Epoch  99 | Loss 0.1052 | Val Acc 0.7800
Epoch 199 | Loss 0.1013 | Val Acc 0.7560
=== Channel=8, Final Test Accuracy: 0.7890 ===
Epoch  99 | Loss 0.0585 | Val Acc 0.7740
Epoch 199 | Loss 0.0235 | Val Acc 0.7800
=== Channel=16, Final Test Accuracy: 0.8050 ===
Epoch  99 | Loss 0.0236 | Val Acc 0.7700
Epoch 199 | Loss 0.0112 | Val Acc 0.7680
=== Channel=32, Final Test Accuracy: 0.7990 ===
Epoch  99 | Loss 0.0148 | Val Acc 0.7860
Epoch 199 | Loss 0.0135 | Val Acc 0.7720
=== Channel=64, Final Test Accuracy: 0.8160 ===
Epoch  99 | Loss 0.0095 | Val Acc 0.7760
Epoch 199 | Loss 0.0076 | Val Acc 0.7760
=== Channel=128, Final Test Accuracy: 0.8170 ===


In [136]:
result

[{'hidden_channels': 8, 'validation_accuracy': 0.756, 'test_accuracy': 0.789},
 {'hidden_channels': 16, 'validation_accuracy': 0.78, 'test_accuracy': 0.805},
 {'hidden_channels': 32, 'validation_accuracy': 0.768, 'test_accuracy': 0.799},
 {'hidden_channels': 64, 'validation_accuracy': 0.772, 'test_accuracy': 0.816},
 {'hidden_channels': 128,
  'validation_accuracy': 0.776,
  'test_accuracy': 0.817}]